# PrivacySweep

**A privacy-awareness prototype for reviewing listing descriptions for possible exposure of Indian mobile-number patterns.**

The notebook combines:
- TF-IDF + Logistic Regression classification
- Indian mobile-number pattern detection
- Predefined public-exposure keyword checks
- Rule-based risk assessment
- User-editable removal-request drafts
- Interactive single-listing and CSV batch analysis dashboards

> **Important:** This prototype analyzes text supplied by the user. It does not crawl websites, verify ownership or public availability of a phone number, or submit removal requests. All findings require human verification.

## Before running

Place `privacy_model.joblib` in the same folder as this notebook. It is the trained scikit-learn pipeline loaded in the model setup section. The model artifact must also be included in the GitHub repository if you want others to run this notebook as-is.


## 1. Install dependencies

In [10]:
%pip install -q ipywidgets scikit-learn pandas joblib

## 2. Imports and load the trained model

In [11]:
import html
import io
import base64
import re
from datetime import datetime
from pathlib import Path

import joblib
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

MODEL_PATH = Path("privacy_model.joblib")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "privacy_model.joblib was not found. Place the trained model file "
        "in the same folder as this notebook."
    )

model = joblib.load(MODEL_PATH)
print(f"Model loaded: {type(model).__name__}")


Model loaded: Pipeline


## 3. Build the single-listing dashboard

In [12]:
#"""
#PrivacySweep dashboard (SmartNet-style design), wired to a real Python workflow.

#Needs:  pip install ipywidgets
#Colab:  from google.colab import output; output.enable_custom_widget_manager()

#HOW IT WORKS
 # The Analyze button runs a Python callback (ipywidgets), not JavaScript, so it can
  #call your ML model directly. Every number on screen comes from what your model
  #returns. Nothing is simulated. Until you connect a model, the dashboard says so
 # and the Analyze button stays disabled.
#"""

import html
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, HTML

# ======================================================================
# 1. CONNECT YOUR MODEL HERE
# ======================================================================
# Write (or import) a function that takes the listing text (str) and returns a dict:
#
#   {
#     "assessment":           "High Review Priority",   # str, final label
#     "mobile_count":         2,                        # int, mobile numbers found
#     "exposure_context":     "Public listing",         # str, how the number is exposed
#     "prediction":           "Review",                 # str, model class label
#     "review_probability":   0.87,                     # float 0-1
#     "no_review_probability":0.13,                     # optional; defaults to 1 - review
#     "high_priority":        True,                     # optional; see threshold below
#   }
#
# Then either set MODEL_FN = your_function below, or call connect_model(your_function)
# in a later cell.
MODEL_FN = None

# Used only when your function does not return "high_priority".
HIGH_PRIORITY_THRESHOLD = 0.5

# ======================================================================
# 2. STATE
# ======================================================================
state = {
    "last": None,       # most recent model result
    "history": [],      # every analysis run in this session
    "drafts": 0,        # incremented only when a draft is actually generated
    "draft_text": "",
    "status": None,     # (message, tone)
}

REQUIRED_KEYS = ["assessment", "mobile_count", "exposure_context",
                 "prediction", "review_probability"]


def _normalize(raw):
    """Validate the model output. Raises a clear error instead of showing bad data."""
    if not isinstance(raw, dict):
        raise ValueError("Model function must return a dict.")
    missing = [k for k in REQUIRED_KEYS if k not in raw]
    if missing:
        raise ValueError("Model output is missing: " + ", ".join(missing))
    p = float(raw["review_probability"])
    if not 0.0 <= p <= 1.0:
        raise ValueError("review_probability must be between 0 and 1.")
    q = float(raw.get("no_review_probability", 1.0 - p))
    high = raw.get("high_priority")
    if high is None:
        high = p >= HIGH_PRIORITY_THRESHOLD
    return {
        "assessment": str(raw["assessment"]),
        "mobile_count": int(raw["mobile_count"]),
        "exposure_context": str(raw["exposure_context"]),
        "prediction": str(raw["prediction"]),
        "review_probability": p,
        "no_review_probability": q,
        "high_priority": bool(high),
    }


def build_removal_draft(result):
    """A plain template. It deliberately does not repeat the phone number."""
    return (
        "Subject: Request to remove personal contact information from a listing\n\n"
        "Hello,\n\n"
        "I am writing to request the removal of a personal mobile number that appears "
        "in the public listing at [LISTING URL].\n\n"
        "The number is shown in the listing description and is not intended for public "
        "use. Please remove it, or edit the listing so it no longer displays this "
        "information, and confirm once this is done.\n\n"
        "Thank you,\n[YOUR NAME]"
    )


# ======================================================================
# 3. STYLE
# ======================================================================
CSS = """
<style>
.ps-root {
    --bg:#120E33; --side:#19134A; --card:#1D1752; --card2:#241D63; --line:#2F2878;
    --text:#F4F2FF; --muted:#9A94D1; --violet:#7C5CFF; --cyan:#38BDF8;
    --green:#4ADE80; --amber:#FBBF24; --red:#FB7185;
    font-family:'Poppins','Segoe UI',Arial,sans-serif; color:var(--text);
    background:var(--bg); border:1px solid var(--line); border-radius:22px;
    overflow:hidden; max-width:1150px; margin:20px auto;
}
.ps-root, .ps-root * { box-sizing:border-box; }
.ps-root .widget-html, .ps-root .widget-html-content { color:var(--text); overflow:visible; }

/* sidebar */
.ps-side { background:var(--side); border-right:1px solid var(--line); padding:22px 16px; }
.ps-logo { display:flex; align-items:center; gap:10px; font-size:19px; font-weight:700; margin-bottom:24px; }
.ps-logo i { width:32px; height:32px; border-radius:9px; font-style:normal;
    background:linear-gradient(135deg,#6D4AFF,#38BDF8); display:flex; align-items:center; justify-content:center; }
.ps-group { color:var(--muted); font-size:12px; margin:18px 0 8px 4px; }
.ps-nav { display:flex; align-items:center; gap:10px; padding:10px 12px; border-radius:9px;
    font-size:13px; color:var(--muted); margin-bottom:4px; }
.ps-nav.active { background:linear-gradient(100deg,#6D4AFF,#8B6CFF); color:#fff; font-weight:600; }
.ps-nav.soon { opacity:.55; cursor:default; }
.ps-nav.soon em { margin-left:auto; font-style:normal; font-size:10px; border:1px solid var(--line);
    border-radius:20px; padding:1px 7px; }
.ps-note { margin-top:24px; padding:14px; border-radius:14px; font-size:12px; line-height:1.5;
    background:linear-gradient(150deg,#6D4AFF,#9A5CFF); }
.ps-note b { display:block; font-size:15px; margin-bottom:4px; }

/* header */
.ps-head h1 { margin:0; font-size:24px; font-weight:600; }
.ps-head p { margin:4px 0 0; color:var(--muted); font-size:12px; }

/* stat cards */
.ps-stats { display:grid; grid-template-columns:repeat(auto-fit,minmax(190px,1fr)); gap:14px; }
.ps-stat { border-radius:14px; padding:16px; min-height:120px; position:relative;
    display:flex; flex-direction:column; justify-content:space-between; }
.ps-stat.c1 { background:linear-gradient(135deg,#6A4CFF,#8F6BFF); }
.ps-stat.c2 { background:linear-gradient(135deg,#2FA8F2,#4F6DF5); }
.ps-stat.c3 { background:linear-gradient(135deg,#A12DE0,#7C4DFF); }
.ps-stat.c4 { background:linear-gradient(135deg,#D63DE8,#A85BFF); }
.ps-stat-label { font-size:13px; font-weight:500; }
.ps-stat-value { font-size:21px; font-weight:700; overflow-wrap:anywhere; }
.ps-stat-sub { font-size:11px; opacity:.85; margin-top:2px; }
.ps-ring { --p:0; position:absolute; right:14px; bottom:14px; width:44px; height:44px; border-radius:50%;
    background:conic-gradient(#fff calc(var(--p)*1%), rgba(255,255,255,.25) 0);
    display:flex; align-items:center; justify-content:center; }
.ps-ring::before { content:""; position:absolute; inset:6px; border-radius:50%; background:rgba(40,24,110,.9); }
.ps-ring span { position:relative; font-size:9px; font-weight:600; }

/* panels */
.ps-panel { background:var(--card); border:1px solid var(--line); border-radius:14px; padding:18px; }
.ps-title { font-size:15px; font-weight:600; margin-bottom:12px; }
.ps-label { display:block; color:var(--muted); font-size:12px; margin-bottom:8px; }
.ps-root textarea { background:var(--bg)!important; color:var(--text)!important;
    border:1px solid var(--line)!important; border-radius:10px!important; font-size:13px!important;
    padding:12px!important; font-family:inherit!important; }
.ps-root textarea:focus { border-color:var(--violet)!important; box-shadow:0 0 0 3px rgba(124,92,255,.25)!important; outline:none; }
.ps-root .ps-primary button { background:linear-gradient(100deg,#6D4AFF,#9A5CFF)!important; color:#fff!important;
    border:none!important; border-radius:10px!important; font-weight:600; font-family:inherit; box-shadow:none!important; }
.ps-root .ps-primary button:disabled { opacity:.5; }
.ps-root .ps-ghost button { background:transparent!important; color:var(--text)!important;
    border:1px solid var(--line)!important; border-radius:9px!important; font-family:inherit; box-shadow:none!important; }
.ps-status { font-size:12px; margin-top:6px; }
.ps-status.amber { color:var(--amber); } .ps-status.green { color:var(--green); }
.ps-status.red { color:var(--red); } .ps-status.blue { color:var(--cyan); }

/* results */
.ps-metric { color:var(--muted); font-size:12px; }
.ps-big { font-size:18px; font-weight:600; color:var(--cyan); margin-top:6px; }
.ps-bar { margin-top:18px; }
.ps-bar-head { display:flex; justify-content:space-between; font-size:12px; color:var(--muted); }
.ps-bar-head strong { color:var(--text); }
.ps-track { background:#2B2470; height:7px; border-radius:20px; margin-top:8px; overflow:hidden; }
.ps-fill { height:100%; border-radius:20px; background:linear-gradient(90deg,#6D4AFF,#38BDF8); }
.ps-fill.alt { background:linear-gradient(90deg,#D946EF,#9A5CFF); }
.ps-removal { background:var(--card2); border:1px solid var(--line); border-radius:12px;
    padding:14px; font-size:12px; line-height:1.6; color:var(--muted); }
.ps-empty { color:var(--muted); font-size:12px; line-height:1.6; padding:30px 8px; text-align:center;
    border:1px dashed var(--line); border-radius:12px; }
.ps-chart { width:100%; height:150px; display:block; }
.ps-foot { text-align:center; color:#6F69A8; font-size:11px; border-top:1px solid var(--line); padding-top:14px; }
</style>
"""

# ======================================================================
# 4. WIDGETS
# ======================================================================
W = widgets
listing_input = W.Textarea(placeholder="Paste a listing description here...",
                           layout=W.Layout(width="100%", height="130px"))
analyze_btn = W.Button(description="Analyze listing", icon="search",
                       layout=W.Layout(width="170px", height="38px"))
analyze_btn.add_class("ps-primary")
new_btn = W.Button(description="New analysis", icon="plus",
                   layout=W.Layout(width="150px", height="36px"))
new_btn.add_class("ps-ghost")
draft_box = W.Textarea(layout=W.Layout(width="100%", height="210px", display="none"))

sidebar = W.HTML(layout=W.Layout(width="200px", flex="0 0 200px"))
sidebar.add_class("ps-side")
head_html = W.HTML("""<div class="ps-head"><h1>Dashboard</h1>
<p>Detect exposure. Protect privacy.</p></div>""")
stats_html = W.HTML()
status_html = W.HTML()
removal_html = W.HTML()
ml_html = W.HTML()
history_html = W.HTML()
footer_html = W.HTML("""<div class="ps-foot">PrivacySweep · Privacy awareness prototype<br>
Results are preliminary and require human verification.
Removal requests are drafted, not automatically submitted.</div>""")

sidebar.value = """
<div class="ps-logo"><i>🛡️</i>PrivacySweep</div>
<div class="ps-group">Menu</div>
<div class="ps-nav active">🏠 Dashboard</div>
<div class="ps-nav soon" aria-disabled="true">📄 Listings <em>Soon</em></div>
<div class="ps-nav soon" aria-disabled="true">🤖 Model results <em>Soon</em></div>
<div class="ps-group">Manage</div>
<div class="ps-nav soon" aria-disabled="true">✉️ Removal requests <em>Soon</em></div>
<div class="ps-nav soon" aria-disabled="true">⚙️ Settings <em>Soon</em></div>
<div class="ps-note"><b>Prototype</b>Items marked Soon are not built yet.
Check every result before sending any request.</div>
"""


def _panel(children, flex):
    box = W.VBox(children, layout=W.Layout(flex=flex, min_width="0"))
    box.add_class("ps-panel")
    return box


input_panel = _panel([
    W.HTML('<div class="ps-title">Analyze a listing</div>'
           '<label class="ps-label">Enter the public listing description</label>'),
    listing_input,
    W.HBox([analyze_btn], layout=W.Layout(margin="12px 0 0")),
    status_html,
], "1.7 1 360px")

removal_panel = _panel([
    W.HTML('<div class="ps-title">Removal request assistant</div>'),
    removal_html,
    draft_box,
], "1 1 260px")

ml_panel = _panel([
    W.HTML('<div class="ps-title">Machine learning results</div>'),
    ml_html,
], "1.7 1 360px")

history_panel = _panel([
    W.HTML('<div class="ps-title">Analysis history</div>'),
    history_html,
], "1 1 260px")

GAP = "16px"
main = W.VBox([
    W.HBox([head_html, new_btn],
           layout=W.Layout(justify_content="space-between", align_items="center", flex_flow="row wrap")),
    stats_html,
    W.HBox([input_panel, removal_panel], layout=W.Layout(flex_flow="row wrap", gap=GAP)),
    W.HBox([ml_panel, history_panel], layout=W.Layout(flex_flow="row wrap", gap=GAP)),
    footer_html,
], layout=W.Layout(flex="1 1 560px", min_width="0", padding="22px 26px 18px", gap=GAP))

root = W.HBox([sidebar, main], layout=W.Layout(flex_flow="row wrap"))
root.add_class("ps-root")


# ======================================================================
# 5. RENDERING (every value below comes from `state`)
# ======================================================================
def _set_status(msg, tone="blue"):
    state["status"] = (msg, tone)
    status_html.value = f'<div class="ps-status {tone}">● {html.escape(msg)}</div>'


def _connection_status():
    if MODEL_FN is None:
        _set_status("Python model connection: not connected. Set MODEL_FN or call connect_model(fn).", "amber")
        analyze_btn.disabled = True
    else:
        name = getattr(MODEL_FN, "__name__", "model")
        _set_status(f"Python model connection: connected ({name})", "green")
        analyze_btn.disabled = False


def _render_stats():
    r = state["last"]
    n = len(state["history"])
    pct = round(100 * state["drafts"] / n) if n else 0
    assess = html.escape(r["assessment"]) if r else "Awaiting analysis"
    count = str(r["mobile_count"]) if r else "—"
    expo = html.escape(r["exposure_context"]) if r else "—"
    stats_html.value = f"""
<div class="ps-stats">
  <div class="ps-stat c1"><div class="ps-stat-label">🔒 Final assessment</div>
    <div class="ps-stat-value">{assess}</div></div>
  <div class="ps-stat c2"><div class="ps-stat-label">📱 Mobile numbers</div>
    <div><div class="ps-stat-value">{count}</div><div class="ps-stat-sub">detected in listing</div></div></div>
  <div class="ps-stat c3"><div class="ps-stat-label">👁️ Public exposure</div>
    <div><div class="ps-stat-value">{expo}</div><div class="ps-stat-sub">how the number appears</div></div></div>
  <div class="ps-stat c4"><div class="ps-stat-label">✉️ Removal drafts</div>
    <div><div class="ps-stat-value">{state['drafts']}</div>
    <div class="ps-stat-sub">of {n} listing{'s' if n != 1 else ''} analyzed</div></div>
    <div class="ps-ring" style="--p:{pct}"><span>{pct}%</span></div></div>
</div>"""


def _render_ml():
    r = state["last"]
    if not r:
        ml_html.value = ('<div class="ps-empty">No model output yet. '
                         'Analyze a listing to see the prediction and probabilities.</div>')
        return
    rp, nq = r["review_probability"] * 100, r["no_review_probability"] * 100
    ml_html.value = f"""
<div class="ps-metric">MODEL PREDICTION</div>
<div class="ps-big">{html.escape(r['prediction'])}</div>
<div class="ps-bar"><div class="ps-bar-head"><span>Review probability</span><strong>{rp:.1f}%</strong></div>
  <div class="ps-track"><div class="ps-fill" style="width:{rp:.1f}%"></div></div></div>
<div class="ps-bar"><div class="ps-bar-head"><span>No review probability</span><strong>{nq:.1f}%</strong></div>
  <div class="ps-track"><div class="ps-fill alt" style="width:{nq:.1f}%"></div></div></div>"""


def _render_history():
    h = state["history"]
    if not h:
        history_html.value = ('<div class="ps-empty">No analyses yet.<br>Each listing you analyze '
                              'adds one bar showing its review probability.</div>')
        return
    recent = h[-12:]
    bw, gap, base, top = 18, 6, 130, 15
    bars = []
    for i, item in enumerate(recent):
        x = 8 + i * (bw + gap)
        ht = max(2, item["p"] * (base - top))
        color = "#D946EF" if item["high"] else "#38BDF8"
        bars.append(f'<rect x="{x}" y="{base - ht:.1f}" width="{bw}" height="{ht:.1f}" rx="3" fill="{color}">'
                    f'<title>{html.escape(item["time"])}: {item["p"]*100:.1f}%</title></rect>')
    ty = base - 0.5 * (base - top)
    history_html.value = f"""
<svg class="ps-chart" viewBox="0 0 310 150" preserveAspectRatio="xMidYMid meet">
  <line x1="0" x2="310" y1="{ty}" y2="{ty}" stroke="#2F2878" stroke-dasharray="4 4"/>
  <text x="2" y="{ty-3}" fill="#9A94D1" font-size="8">50%</text>
  {''.join(bars)}
</svg>
<div class="ps-metric">Review probability per analysis (last {len(recent)}). Pink = High Review Priority.</div>"""



def _render_removal():
    r = state["last"]

    if not r:
        msg = (
            "No removal request generated. A draft is prepared only when a listing "
            "receives High Review Priority."
        )
        draft_box.layout.display = "none"

    elif r["high_priority"]:
        msg = (
            "Draft prepared below. Edit it, fill in the placeholders, and send it yourself. "
            "PrivacySweep never submits it."
        )
        draft_box.value = state["draft_text"]
        draft_box.layout.display = ""

    else:
        msg = "No draft needed. This listing did not receive High Review Priority."
        draft_box.layout.display = "none"

    removal_html.value = (
        f'<div class="ps-removal">✉️ {html.escape(msg)}</div>'
    )


# ======================================================================
# 6. ACTIONS
# ======================================================================

# Refresh all dashboard sections

def _render_all():
    _render_stats()
    _render_ml()
    _render_history()
    _render_removal()
    
def on_analyze(_):
    text = listing_input.value.strip()

    if not text:
        _set_status("Enter a listing description first.", "amber")
        return

    if MODEL_FN is None:
        _connection_status()
        return

    analyze_btn.disabled = True
    _set_status("Running model...", "blue")

    try:
        raw_result = MODEL_FN(text)
        result = _normalize(raw_result)

        # Update the latest result using the current input only
        state["last"] = result

        # Add this analysis to session history
        state["history"].append({
            "time": datetime.now().strftime("%H:%M:%S"),
            "p": result["review_probability"],
            "high": result["high_priority"],
        })

        # Generate a draft only for high-priority results
        if result["high_priority"]:
            state["draft_text"] = build_removal_draft(result)
            state["drafts"] += 1
        else:
            state["draft_text"] = ""

        # Refresh every dashboard section
        _render_all()

        _set_status("Analysis completed successfully.", "green")

    except Exception as e:
        _set_status(
            f"Model error: {type(e).__name__}: {e}",
            "red"
        )

    finally:
        analyze_btn.disabled = MODEL_FN is None



def on_new(_):
    """Clear the current listing and result. Session history and draft count are kept."""
    listing_input.value = ""
    state["last"] = None
    state["draft_text"] = ""
    draft_box.value = ""
    _render_all()
    _connection_status()


def connect_model(fn):
    """Call this with your model function, e.g. connect_model(my_predict)."""
    global MODEL_FN
    MODEL_FN = fn
    _connection_status()


analyze_btn.on_click(on_analyze)
new_btn.on_click(on_new)

_render_all()
_connection_status()
display(HTML(CSS))

## 4. Detection and risk-assessment logic

In [4]:
import re

def privacy_predict(description):
    description_lower = description.lower()

    # 1. Detect Indian mobile-number patterns
    mobile_numbers = re.findall(
        r'(?<!\d)[6-9]\d{9}(?!\d)',
        description
    )

    # 2. Detect public-exposure context
    exposure_keywords = [
        "public directory",
        "public listing",
        "public website",
        "public profile",
        "visible publicly",
        "listed online",
        "exposed online",
        "available online",
        "search results",
        "business directory"
    ]

    negative_exposure_phrases = [
        "no public listing",
        "no public website exposure",
        "no public exposure",
        "not publicly listed",
        "not exposed publicly",
        "not available online",
        "no exposure online"
    ]

    negative_exposure = any(
        phrase in description_lower
        for phrase in negative_exposure_phrases
    )

    exposure_found = (
        any(keyword in description_lower for keyword in exposure_keywords)
        and not negative_exposure
    )

    # 3. Get ML prediction and probabilities
    prediction = model.predict([description])[0]
    probabilities = model.predict_proba([description])[0]
    classes = list(model.classes_)


    review_probability = (
        float(probabilities[classes.index("Review")])
        if "Review" in classes else 0.0
    )

    no_review_probability = (
        float(probabilities[classes.index("No Review")])
        if "No Review" in classes else 0.0
    )

    # 4. Apply final risk-assessment rules
    if mobile_numbers and exposure_found:
        assessment = "High Review Priority"
        exposure_context = "Public exposure detected"
        high_priority = True

    elif mobile_numbers:
        assessment = "Needs Verification"
        exposure_context = "Not detected"
        high_priority = False

    else:
        assessment = str(prediction)
        exposure_context = "Not detected"
        high_priority = False

    return {
        "assessment": assessment,
        "mobile_count": len(mobile_numbers),
        "exposure_context": exposure_context,
        "prediction": str(prediction),
        "review_probability": review_probability,
        "no_review_probability": no_review_probability,
        "high_priority": high_priority
    }


# Connect the prediction function to the dashboard.
connect_model(privacy_predict)
print("PrivacySweep prediction function connected.")


PrivacySweep prediction function connected.


## 5. CSV batch analysis

In [13]:

# CSV batch-analysis controls and callbacks
batch_df = None
batch_results = None

batch_upload = widgets.FileUpload(
    accept=".csv",
    multiple=False,
    description="Upload CSV"
)

batch_column = widgets.Dropdown(
    options=[],
    description="Text column:",
    disabled=True,
    layout=widgets.Layout(width="350px")
)

batch_analyze_btn = widgets.Button(
    description="Analyze CSV",
    icon="search",
    disabled=True,
    layout=widgets.Layout(width="160px")
)

batch_download_btn = widgets.Button(
    description="Download Results",
    icon="download",
    disabled=True,
    layout=widgets.Layout(width="190px")
)

batch_status = widgets.HTML(
    value="<span style='color:#B9B3E8'>Upload a CSV file to begin.</span>"
)
batch_output = widgets.Output()


def _uploaded_file_bytes(upload_value):
    """Return bytes from ipywidgets 8 FileUpload value."""
    if isinstance(upload_value, dict):
        if not upload_value:
            raise ValueError("No uploaded file was found.")
        uploaded_file = next(iter(upload_value.values()))
    elif isinstance(upload_value, (tuple, list)):
        if not upload_value:
            raise ValueError("No uploaded file was found.")
        uploaded_file = upload_value[0]
    else:
        raise ValueError("Unsupported upload format.")

    content = uploaded_file["content"]
    if isinstance(content, memoryview):
        content = content.tobytes()
    return bytes(content)


def on_batch_upload(change):
    """Read an uploaded CSV and populate the text-column selector."""
    global batch_df, batch_results

    if not batch_upload.value:
        return

    try:
        raw_bytes = _uploaded_file_bytes(batch_upload.value)
        decoded = None
        last_error = None

        encodings = (
            ["utf-16", "utf-8-sig", "cp1252", "latin1"]
            if raw_bytes.startswith((b"\\xff\\xfe", b"\\xfe\\xff"))
            else ["utf-8-sig", "cp1252", "latin1"]
        )

        for encoding in encodings:
            try:
                decoded = raw_bytes.decode(encoding)
                break
            except UnicodeDecodeError as exc:
                last_error = exc

        if decoded is None:
            raise ValueError(f"Could not decode the CSV file: {last_error}")

        try:
            loaded_df = pd.read_csv(io.StringIO(decoded), sep=None, engine="python")
        except Exception:
            loaded_df = pd.read_csv(io.StringIO(decoded))

        if loaded_df.empty:
            raise ValueError("The CSV file contains no records.")
        if len(loaded_df.columns) == 0:
            raise ValueError("The CSV file has no columns.")

        batch_df = loaded_df
        batch_results = None
        batch_download_btn.disabled = True
        batch_column.options = list(batch_df.columns)

        # Prefer a column named description; otherwise use the first column.
        preferred = next(
            (column for column in batch_df.columns
             if str(column).strip().lower() in {"description", "listing_description", "text"}),
            batch_df.columns[0]
        )
        batch_column.value = preferred
        batch_column.disabled = False
        batch_analyze_btn.disabled = False

        batch_status.value = (
            f"<span style='color:#4ADE80'>Loaded {len(batch_df)} records. "
            f"Selected column: {html.escape(str(preferred))}</span>"
        )

        with batch_output:
            batch_output.clear_output()

    except Exception as exc:
        batch_df = None
        batch_results = None
        batch_column.options = []
        batch_column.disabled = True
        batch_analyze_btn.disabled = True
        batch_download_btn.disabled = True
        batch_status.value = (
            f"<span style='color:#FB7185'>Upload error: {html.escape(str(exc))}</span>"
        )


def on_batch_analyze(_):
    """Analyze each row in the selected text column."""
    global batch_results

    if batch_df is None:
        batch_status.value = "<span style='color:#FBBF24'>Upload a CSV file first.</span>"
        return

    text_column = batch_column.value
    if text_column not in batch_df.columns:
        batch_status.value = "<span style='color:#FBBF24'>Select a valid text column.</span>"
        return

    if MODEL_FN is None:
        batch_status.value = "<span style='color:#FB7185'>Model is not connected.</span>"
        return

    batch_analyze_btn.disabled = True
    batch_download_btn.disabled = True

    try:
        results = []

        for value in batch_df[text_column]:
            description = "" if pd.isna(value) else str(value).strip()

            if not description:
                result = {
                    "assessment": "Needs Verification",
                    "mobile_count": 0,
                    "exposure_context": "Empty description",
                    "prediction": "Not analyzed",
                    "review_probability": None,
                    "no_review_probability": None,
                    "high_priority": False
                }
            else:
                result = _normalize(MODEL_FN(description))

            results.append(result)

        batch_results = batch_df.copy()
        batch_results["Assessment"] = [r["assessment"] for r in results]
        batch_results["Mobile Numbers Found"] = [r["mobile_count"] for r in results]
        batch_results["Exposure Context"] = [r["exposure_context"] for r in results]
        batch_results["ML Prediction"] = [r["prediction"] for r in results]
        batch_results["Review Probability (%)"] = [
            round(r["review_probability"] * 100, 2)
            if r["review_probability"] is not None else None
            for r in results
        ]
        batch_results["High Priority"] = [r["high_priority"] for r in results]

        total = len(results)
        high_count = sum(r["assessment"] == "High Review Priority" for r in results)
        verify_count = sum(r["assessment"] == "Needs Verification" for r in results)
        no_review_count = sum(r["assessment"] == "No Review" for r in results)

        with batch_output:
            batch_output.clear_output()
            display(HTML(f"""
            <div style="display:flex;flex-wrap:wrap;gap:12px;margin:14px 0;color:#fff">
              <div style="background:#251b60;padding:16px;border-radius:12px">
                <b>Total analyzed</b><h2>{total}</h2>
              </div>
              <div style="background:#6b215e;padding:16px;border-radius:12px">
                <b>High priority</b><h2>{high_count}</h2>
              </div>
              <div style="background:#59421b;padding:16px;border-radius:12px">
                <b>Needs verification</b><h2>{verify_count}</h2>
              </div>
              <div style="background:#174b43;padding:16px;border-radius:12px">
                <b>No review</b><h2>{no_review_count}</h2>
              </div>
            </div>
            """))
            display(batch_results)

        batch_status.value = (
            f"<span style='color:#4ADE80'>Successfully analyzed {total} records.</span>"
        )
        batch_download_btn.disabled = False

    except Exception as exc:
        batch_results = None
        batch_status.value = (
            f"<span style='color:#FB7185'>Analysis error: "
            f"{html.escape(str(exc))}</span>"
        )
    finally:
        batch_analyze_btn.disabled = False


def on_batch_download(_):
    """Display a browser-download link for the analyzed CSV."""
    if batch_results is None:
        batch_status.value = "<span style='color:#FBBF24'>Analyze the CSV before downloading.</span>"
        return

    csv_bytes = batch_results.to_csv(index=False).encode("utf-8-sig")
    encoded = base64.b64encode(csv_bytes).decode("ascii")
    link = f"""
    <a download="PrivacySweep_Batch_Results.csv"
       href="data:text/csv;base64,{encoded}"
       style="display:inline-block;background:#7C5CFF;color:white;padding:11px 16px;
              border-radius:8px;text-decoration:none;font-weight:600">
       Download PrivacySweep Batch Results
    </a>
    """
    with batch_output:
        display(HTML(link))


batch_upload.observe(on_batch_upload, names="value")
batch_analyze_btn.on_click(on_batch_analyze)
batch_download_btn.on_click(on_batch_download)

print("CSV batch-analysis controls are ready.")


CSV batch-analysis controls are ready.


## 6. Unified dashboard and navigation

In [14]:

# Unified application navigation
PURPLE = "#7C5CFF"
DARK = "#120E33"
SIDEBAR_COLOR = "#19134A"
CARD = "#1D1752"
CYAN = "#38BDF8"

# The original dashboard already has a sidebar. Hide it here because
# this unified navigation replaces its static labels.
sidebar.layout.display = "none"
root.children = (main,)

batch_header = widgets.HTML("""
<div style="margin-bottom:16px">
  <h2 style="color:#fff;margin:0">CSV Batch Analysis</h2>
  <p style="color:#B9B3E8">Upload a CSV, select the listing-description column,
  analyze multiple records, and download the results.</p>
</div>
""")

batch_panel = widgets.VBox([
    batch_header,
    batch_upload,
    batch_column,
    widgets.HBox([batch_analyze_btn, batch_download_btn],
                 layout=widgets.Layout(gap="10px")),
    batch_status,
    batch_output
], layout=widgets.Layout(width="100%", gap="12px"))

def make_model_results():
    result = state.get("last")
    if result is None:
        body = "<p style='color:#B9B3E8'>Analyze a listing to view its latest model results.</p>"
    else:
        body = f"""
        <div style="display:grid;grid-template-columns:repeat(2,minmax(150px,1fr));gap:12px">
          <div style="background:{CARD};padding:16px;border-radius:12px">
            <span style="color:#B9B3E8">Assessment</span>
            <h3 style="color:white">{html.escape(result['assessment'])}</h3>
          </div>
          <div style="background:{CARD};padding:16px;border-radius:12px">
            <span style="color:#B9B3E8">ML prediction</span>
            <h3 style="color:white">{html.escape(result['prediction'])}</h3>
          </div>
          <div style="background:{CARD};padding:16px;border-radius:12px">
            <span style="color:#B9B3E8">Review score</span>
            <h3 style="color:{CYAN}">{result['review_probability']:.2%}</h3>
          </div>
          <div style="background:{CARD};padding:16px;border-radius:12px">
            <span style="color:#B9B3E8">Mobile patterns found</span>
            <h3 style="color:white">{result['mobile_count']}</h3>
          </div>
        </div>
        <p style="color:#B9B3E8">Exposure context: {html.escape(result['exposure_context'])}</p>
        <p style="color:#B9B3E8">The ML score is not proof that a number is publicly exposed.</p>
        """
    return widgets.HTML(
        "<div style='color:white'><h2>Model Results</h2>" + body + "</div>"
    )

def make_removal_requests():
    draft = state.get("draft_text", "")
    if not draft:
        return widgets.HTML("""
        <div style="color:white">
          <h2>Removal Requests</h2>
          <p style="color:#B9B3E8">No draft is currently available.
          A draft is generated only for a High Review Priority result.</p>
        </div>
        """)
    editor = widgets.Textarea(
        value=draft,
        layout=widgets.Layout(width="100%", height="240px")
    )
    return widgets.VBox([
        widgets.HTML("""
          <div style="color:white">
            <h2>Removal Requests</h2>
            <p style="color:#B9B3E8">Review and edit this draft. PrivacySweep does not submit it.</p>
          </div>
        """),
        editor
    ])

settings_panel = widgets.HTML("""
<div style="color:white">
  <h2>Settings</h2>
  <div style="background:#1D1752;padding:18px;border-radius:12px;color:#B9B3E8">
    Settings and preferences — Coming soon.
  </div>
</div>
""")

page_area = widgets.VBox(
    [],
    layout=widgets.Layout(width="100%", padding="22px", min_height="650px")
)

nav_buttons = {}
nav_specs = [
    ("Dashboard", "▦"),
    ("Batch Analysis", "▤"),
    ("Model Results", "◈"),
    ("Removal Requests", "✉"),
    ("Settings", "⚙")
]

brand = widgets.HTML("""
<div style="padding:14px 10px 24px">
  <div style="font-size:23px;font-weight:800;color:white">
    ◈ Privacy<span style="color:#7C5CFF">Sweep</span>
  </div>
  <div style="font-size:10px;letter-spacing:2px;color:#B9B3E8">
    PRIVACY PROTECTION
  </div>
</div>
""")

nav_sidebar = widgets.VBox(
    [brand],
    layout=widgets.Layout(
        width="220px", min_width="220px", padding="12px",
        background=SIDEBAR_COLOR
    )
)

for label, symbol in nav_specs:
    button = widgets.Button(
        description=f"{symbol}   {label}",
        layout=widgets.Layout(width="100%", height="44px"),
        style={"button_color": SIDEBAR_COLOR, "font_weight": "500"}
    )
    nav_buttons[label] = button
    nav_sidebar.children += (button,)

def show_page(page):
    for label, button in nav_buttons.items():
        button.style.button_color = PURPLE if label == page else SIDEBAR_COLOR

    pages = {
        "Dashboard": (root,),
        "Batch Analysis": (batch_panel,),
        "Model Results": (make_model_results(),),
        "Removal Requests": (make_removal_requests(),),
        "Settings": (settings_panel,)
    }
    page_area.children = pages[page]

for label, button in nav_buttons.items():
    button.on_click(lambda _, selected=label: show_page(selected))

app_header = widgets.HTML("""
<div style="padding:15px 22px;background:#19134A;border-bottom:1px solid #342A70">
  <div style="font-size:20px;font-weight:700;color:white">Privacy Intelligence Dashboard</div>
  <div style="font-size:12px;color:#B9B3E8">
    Detect exposure · Review risk · Prepare user-submitted removal requests
  </div>
</div>
""")

application = widgets.VBox([
    app_header,
    widgets.HBox(
        [nav_sidebar, page_area],
        layout=widgets.Layout(width="100%", align_items="stretch", gap="0")
    )
], layout=widgets.Layout(width="100%"))

display(HTML("""
<style>
.jp-OutputArea-output {background:#120E33 !important;}
.widget-button button, .widget-button {
  background:#2A205F !important;color:#fff !important;
  border:1px solid #51418F !important;border-radius:8px !important;
}
.widget-button:hover {background:#7C5CFF !important;}
.widget-text input,.widget-textarea textarea,.widget-dropdown select {
  background:#19134A !important;color:#fff !important;
  border:1px solid #51418F !important;border-radius:8px !important;
}
.widget-label {color:#F4F0FF !important;}
</style>
"""))

show_page("Dashboard")
display(application)


## 7. Quick validation examples

In [15]:

# Quick functional checks (does not change dashboard state)
test_descriptions = [
    "Mobile number 7898021395 visible on public directory.",
    "Business opening hours displayed on the website.",
    "Customer contact number 7898021395 was mentioned in an internal note. No public listing or website exposure is indicated.",
    "No personal contact details visible."
]

test_results = pd.DataFrame([
    {
        "Description": description,
        "Assessment": privacy_predict(description)["assessment"],
        "Mobile Numbers Found": privacy_predict(description)["mobile_count"],
        "Review Probability (%)": round(
            privacy_predict(description)["review_probability"] * 100, 2
        )
    }
    for description in test_descriptions
])

display(test_results)


,Description,Assessment,Mobile Numbers Found,Review Probability (%)
0,Mobile number 7898021395 visible on public dir...,High Review Priority,1,55.00
1,Business opening hours displayed on the website.,No Review,0,36.39
2,Customer contact number 7898021395 was mention...,Needs Verification,1,44.62
3,No personal contact details visible.,No Review,0,34.65


In [16]:
from IPython.display import display, HTML

display(HTML("""
<style>
.jp-OutputArea-output,
.jp-RenderedText,
.jp-OutputArea-output pre {
    background-color: #FFFFFF !important;
    color: #222222 !important;
}

.jp-RenderedHTMLCommon {
    background-color: #FFFFFF !important;
    color: #222222 !important;
}

.jp-OutputArea-output table,
.jp-OutputArea-output td,
.jp-OutputArea-output th {
    background-color: #FFFFFF !important;
    color: #222222 !important;
}
</style>
"""))

print("Notebook outputs are now readable.")

Notebook outputs are now readable.


In [9]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("\nAvailable files:")

for file in Path.cwd().iterdir():
    print(file.name)

Current folder: /drive/notebooks

Available files:
Dashboard 2.ipynb
Intro.ipynb
PrivacySweep_Clean_GitHub.ipynb
PrivacySweep_Dashboard
Privacysweep.ipynb
Untitled.ipynb
Untitled5.ipynb
Untitled6.ipynb
app.py
final dashbaord.ipynb
privacy_model.joblib
Mandelbrot.ipynb
cpp.ipynb
cpp-third-party-libs.ipynb
Lorenz.ipynb
cpp-tiny-ray-tracer.ipynb
r.ipynb
sqlite.ipynb


## Project limitations

- The classifier uses a small synthetic training dataset and is a prototype, not a validated production model.
- The phone-number detector identifies strings matching an Indian mobile-number pattern; it does not verify number ownership or validity.
- Exposure detection relies on predefined keywords and a limited set of negative phrases. It may miss paraphrases or misinterpret context.
- PrivacySweep does not search or crawl public websites and directories.
- Removal-request text is a draft only. The user must verify the listing and submit the request themselves.
- Model probabilities are classifier scores, not calibrated probabilities of real-world exposure.

**Use:** Treat results as prompts for human review, not as proof of a privacy violation.
